# Make the splits

A split decides **which charts to keep** and **which part (train/validation/test) each one goes to**. It scans every `.tja` under `raw/`, drops charts the parser can't handle or the filters exclude, and splits the rest by group, so every chart of a group lands in the same part.

This notebook makes the two splits used from here on:

| Split | Courses | Grouping | Parts |
|---|---|---|---|
| `oni_v2` | Oni | song folder (keeps covers together) | stratified by star level, 80/10/10, seed 42 |
| `all_v1` | all five | song folder | copied from `oni_v2`; songs without an Oni chart go to train |

So `all_v1` (for pretraining on all courses) never puts an `oni_v2` validation or test song in train, in any course.

Output goes to `splits/<name>/`: `config.json`, `manifest.csv` and `report.json`. Splits are never edited in place. If you change a filter, give the split a new name. `oni_v1` is left as it was.

In [ ]:
from donderformer.paths import DataPaths
from donderformer.data.data_splitter import DataSplitter

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for the toy dataset.
paths = DataPaths()

oni_splitter = DataSplitter(
    name="oni_v2",
    courses=("oni",),          # None = all; options: easy, normal, hard, oni, ura
    genres=None,               # None = all; top-level ESE folders, e.g. ("01 Pop", "02 Anime")
    bpm_range=None,            # e.g. (100, 220)
    level_range=None,          # e.g. (7, 10)
    ratios=(0.8, 0.1, 0.1),
    seed=42,
    group_by="song",           # "song" = <genre>/<song>/ folder; "audio" = one audio file (oni_v1)
    stratify_by_level=True,    # strata: <=6, 7, 8, 9, 10 stars (by the song's Oni level)
)

all_splitter = DataSplitter(
    name="all_v1",
    courses=None,
    ratios=(0.8, 0.1, 0.1),
    seed=42,
    group_by="song",
    song_parts_from="oni_v2",  # same validation/test songs as oni_v2
)

In [ ]:
oni_report = oni_splitter.run(paths)

In [ ]:
all_report = all_splitter.run(paths)

## Survival funnel

Charts left after each filter, in order. A chart dropped for several reasons is counted at the first one. Charts that later fail labelling (e.g. notes past the end of the audio) are listed by `02_extract_features` in `features/<extractor>/failures/<split>.json`.

In [ ]:
import json

def show_funnel(report, course=None):
    steps = report["funnel_by_course"][course] if course else report["funnel"]
    print(f"{report['name']}{' / ' + course if course else ''}")
    for step in steps:
        print(f"  {step['stage']:<26} -{step['dropped']:>6}  {step['remaining']:>6}")

show_funnel(all_report)                    # every course, all filters
show_funnel(all_report, "oni")
show_funnel(oni_report)                    # oni_v2: course filter applied

## Leak checks

`run()` already asserts that no song folder or audio file is in two parts, and for `all_v1` that every song kept the part it has in `oni_v2`. Titles found in more than one part are listed for a manual look: they're usually different songs with the same name.

In [ ]:
for report in (oni_report, all_report):
    print(report["name"], {k: v for k, v in report["leak_check"].items() if k != "titles_crossing"})
    for title, parts in report["leak_check"]["titles_crossing"].items():
        print("   same title in", parts, ":", title)
print(all_report["song_parts_from"])

## Compare splits

Put any existing splits in `SPLITS`. Each figure has one row per split and the same plot in every column, so the splits line up; within a split, `all` is every kept chart and the other bars are its parts. Note density needs the audio durations cached by `00_dataset_analysis` (`COMPUTE_DURATIONS = True`). Figures are saved under `data/meta/split_eda/`.

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from donderformer.data.chart_stats import NOTE_TYPES, audio_durations, densities, note_counts
from donderformer.data.data_splitter import PARTS, read_manifest
from donderformer.data.tja import COURSES

GROUPS = ("all",) + PARTS
DURATIONS = paths.meta / "audio_durations.csv"   # made by 00_dataset_analysis


def song(row):
    return row["tja"].rsplit("/", 1)[0]


def load_split(name):
    rows = read_manifest(paths, name)
    report = json.loads((paths.split_dir(name) / "report.json").read_text())
    return {"all": rows, **{p: [r for r in rows if r["split"] == p] for p in PARTS}}, report


def pct(rows, key, categories):
    return [100 * sum(r[key] == c for r in rows) / max(len(rows), 1) for c in categories]


def grouped_bars(ax, categories, values, horizontal=False):
    """values: label -> one number per category."""
    width = 0.8 / len(values)
    x = np.arange(len(categories))
    for i, (label, v) in enumerate(values.items()):
        offset = x + (i - (len(values) - 1) / 2) * width
        (ax.barh if horizontal else ax.bar)(offset, v, width, label=label)
    if horizontal:
        ax.set_yticks(x, categories)
        ax.invert_yaxis()
    else:
        ax.set_xticks(x, categories, rotation=30)


def step_hists(ax, groups, values_of, bins):
    for g, rows in groups.items():
        values = [v for v in map(values_of, rows) if v is not None]
        if values:
            ax.hist(values, bins=bins, histtype="step", weights=np.full(len(values), 100 / len(values)), label=g)


def grid(names, columns, title):
    fig, axes = plt.subplots(len(names), len(columns), figsize=(4.6 * len(columns), 3.4 * len(names)), squeeze=False)
    for ax, column in zip(axes[0], columns):
        ax.set_title(column)
    for row_axes, name in zip(axes, names):
        row_axes[0].annotate(name, xy=(-0.3, 0.5), xycoords="axes fraction", rotation=90,
                             ha="center", va="center", fontsize=13, weight="bold")
    fig.suptitle(title)
    return fig, axes


def finish(fig, axes, save_as, shared_y=()):
    """Same y range down each column in ``shared_y``, so rows compare directly."""
    for col in shared_y:
        top = max(ax.get_ylim()[1] for ax in axes[:, col])
        for ax in axes[:, col]:
            ax.set_ylim(0, top)
    for ax in axes[0]:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    fig.tight_layout()
    out = paths.meta / "split_eda"
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{save_as}.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def compare_splits(names):
    splits = {n: load_split(n) for n in names}
    every = [r for groups, _ in splits.values() for r in groups["all"]]
    courses = [c for c in COURSES if any(r["course"] == c for r in every)]
    genres = sorted({r["genre"] for r in every})
    density = densities(every, audio_durations(paths.raw, {r["audio"] for r in every}, DURATIONS))
    lost = Counter()
    for _, report in splits.values():
        lost.update(dict(list(report["lost_reasons"].items())[:8]))
    reasons = [r for r, _ in lost.most_common(10)]
    tag = "+".join(names)

    for name, (groups, report) in splits.items():
        sizes = ", ".join(f"{p} {len({song(r) for r in groups[p]})}/{len(groups[p])}" for p in PARTS)
        leaks = {k: v for k, v in report.get("leak_check", {}).items() if k != "titles_crossing"}
        print(f"{name}: songs/charts {sizes}; leak check {leaks or 'n/a'}")

    columns = ["Songs per part", "Charts per part", "Course share (%)", "Genre share (%)",
               "Note types (% of events)", "Note density (% of charts)", "BPM (% of charts)", "Lost reasons (charts)"]
    fig, axes = grid(names, columns, "Split overview")
    bpm_bins = np.histogram_bin_edges([float(r["bpm"]) for r in every], bins=30)
    density_bins = np.histogram_bin_edges(list(density.values()) or [0, 1], bins=30)
    for row_axes, (name, (groups, report)) in zip(axes, splits.items()):
        row_axes[0].bar(PARTS, [len({song(r) for r in groups[p]}) for p in PARTS])
        row_axes[1].bar(PARTS, [len(groups[p]) for p in PARTS])
        grouped_bars(row_axes[2], courses, {g: pct(rows, "course", courses) for g, rows in groups.items()})
        grouped_bars(row_axes[3], genres, {g: pct(rows, "genre", genres) for g, rows in groups.items()}, horizontal=True)

        counts = np.array([note_counts(rows) for rows in groups.values()], dtype=float)
        shares = 100 * counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        bottom = np.zeros(len(groups))
        for i, label in enumerate(NOTE_TYPES):
            row_axes[4].bar(list(groups), shares[:, i], bottom=bottom, label=label)
            bottom += shares[:, i]

        if density:
            step_hists(row_axes[5], groups, lambda r: density.get(r["chart_id"]), density_bins)
        else:
            row_axes[5].text(0.5, 0.5, "no audio durations:\nrun 00 with COMPUTE_DURATIONS",
                             ha="center", va="center", transform=row_axes[5].transAxes)
        step_hists(row_axes[6], groups, lambda r: float(r["bpm"]), bpm_bins)
        row_axes[7].barh(reasons[::-1], [report["lost_reasons"].get(r, 0) for r in reasons[::-1]])
    finish(fig, axes, tag, shared_y=(2, 4, 5, 6))

    # Star levels within each course, and mean density per level.
    fig, axes = grid(names, [f"{c}: level share (%)" for c in courses], "Star levels")
    for row_axes, (name, (groups, _)) in zip(axes, splits.items()):
        for ax, course in zip(row_axes, courses):
            levels = sorted({r["level"] for r in every if r["course"] == course},
                            key=lambda lv: (lv == "", float(lv or 0)))
            by_group = {g: [r for r in rows if r["course"] == course] for g, rows in groups.items()}
            if by_group["all"]:
                grouped_bars(ax, [lv or "?" for lv in levels], {g: pct(rows, "level", levels) for g, rows in by_group.items()})
            else:
                ax.set_axis_off()
    finish(fig, axes, f"{tag}_levels", shared_y=range(len(courses)))

    if density:
        fig, axes = grid(names, [f"{c}: mean density by level" for c in courses], "Note density by level")
        for row_axes, (name, (groups, _)) in zip(axes, splits.items()):
            for ax, course in zip(row_axes, courses):
                if not any(r["course"] == course for r in groups["all"]):
                    ax.set_axis_off()
                    continue
                for g, rows in groups.items():
                    by_level = {}
                    for r in rows:
                        if r["course"] == course and r["level"] and r["chart_id"] in density:
                            by_level.setdefault(r["level"], []).append(density[r["chart_id"]])
                    levels = sorted(by_level, key=float)
                    ax.plot(levels, [np.mean(by_level[lv]) for lv in levels], marker="o", label=g)
        finish(fig, axes, f"{tag}_density", shared_y=range(len(courses)))

In [ ]:
SPLITS = ["oni_v2", "all_v1"]   # any splits under splits/, e.g. add "oni_v1" or "all_courses_eda_v1"
compare_splits(SPLITS)

## Observations on `all_courses_eda_v1`

Written by Pierre for his all-course split (grouped by audio file, not stratified). Add it to `SPLITS` to see the plots these refer to.

The split contains **1,591 / 198 / 200 songs** and **4,989 / 627 / 604 charts** for train / validation / test. The 80/10/10 target applies to songs, so chart counts do not have to follow exactly the same proportions.

### Difficulty categories and star levels

The main difficulty categories have similar proportions across splits. Easy and Normal each account for about 22% of the charts, Hard for 24–26%, and Oni for about 28%. Ura is less common: it represents 3.57% of training charts but only 2.32% of test charts.

Star levels describe variation within a difficulty category. Matching the number of Normal or Oni charts alone would not guarantee a comparable distribution of levels. Some differences remain: Hard level 6 accounts for 30.1% of test Hard charts versus 22.3% in training. Oni level 6 is less frequent in test (10.5%) than in training (16.2%).

Rare groups need care. The test set contains only **14 Ura charts**, including 8 at level 10. Results for this group may depend strongly on the particular songs selected. The single Easy level 6 chart is in validation and absent from training; its metadata should be checked before interpreting it as a representative group.

### Note types

Note type proportions are similar across splits. Don and Ka together account for about **91% of note events**. This imbalance is mainly a property of the retained dataset, rather than a large change introduced by the split. Similar overall proportions do not guarantee that every difficulty and level has the same note mix.

### Note density

| Split | Charts | Mean density (events/s) | Median density (events/s) |
|---|---:|---:|---:|
| Train | 4,989 | 2.805 | 2.451 |
| Validation | 627 | 2.813 | 2.445 |
| Test | 604 | 2.876 | 2.576 |

The density histograms broadly overlap. Test density is slightly higher on average, but the plots do not show a large overall distribution shift. This is a descriptive comparison, not a statistical equivalence test.

### Conclusion

The current split broadly preserves difficulty categories, note type proportions and overall density. These checks do not provide a strong reason to replace it immediately. The main limitations are the small Ura test set, rare levels and parser filtering. Any later change to improve coverage should keep all charts sharing an audio file together and use a new split name. The audio-path overlap check does not detect duplicate recordings stored under different paths.
